# PWL Approximation (SOS2) — Before, Principle, Application, and Effect

If a non-convex, single-variable function (like an efficiency curve or multimodal cost curve) is incorporated directly into an optimization, it becomes a non-convex constraint requiring spatial branching. In practice, it is handled via **Piecewise Linear (PWL) approximation**, but if Big-M + binaries are used to select the segment, the number of binary variables increases proportionally to the number of segments.

This notebook tracks PWL approximation using the following pattern:

1. **Issue (before)** — Non-convex functions and the fact that Big-M approximation requires a large number of binaries
2. **Principle (principle)** — Breakpoints and approximation error, and how SOS2 represents adjacency without binaries
3. **Application (how)** — Apply `mk.pwl_sos2` in a single line
4. **Effect (before/after)** — Compare SOS2 vs Big-M in terms of model scale and solving

The subject is `samples/physics_and_control_minlp/pwl_sos.py`. We minimize a non-convex, multimodal function $f(x)=\sin(1.5x)+0.15(x-5)^2$ using PWL approximation.
You can switch between the SOS2 version and Big-M version via `build_model(method="sos2" | "bigm")`.

In [ ]:
import sys, pathlib
# リポジトリルート(pyproject.toml を持つ階層)を探す。docs/samples/ があるため samples 有無では docs で止まる。
ROOT = pathlib.Path.cwd()
while not (ROOT / "pyproject.toml").is_file() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
for sub in ["samples", "samples/physics_and_control_minlp"]:
    p = str(ROOT / sub)
    if p not in sys.path:
        sys.path.insert(0, p)

import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from IPython.display import HTML, display

def show(fig):  # 静的サイトに埋め込める形でグラフを表示(CDN の plotly.js を読む)
    display(HTML(fig.to_html(include_plotlyjs="cdn", full_html=False,
                             config={"displayModeBar": False})))

import minlpkit as mk
import pwl_sos as pw

C = dict(ink="#0b0b0b", ink2="#52514e", muted="#898781", grid="#e1e0d9",
         axis="#c3c2b7", surface="#fcfcfb", s1="#2a78d6", s2="#008300", warn="#c25a00")
FONT = 'system-ui, -apple-system, "Segoe UI", sans-serif'

def base_layout(title, xtitle, ytitle, height=380):
    ax = dict(gridcolor=C["grid"], linecolor=C["axis"],
              tickfont=dict(color=C["muted"], size=11),
              title_font=dict(color=C["ink2"], size=12), zeroline=False)
    return go.Layout(
        title=dict(text=title, font=dict(color=C["ink"], size=15, family=FONT), x=0.01),
        paper_bgcolor=C["surface"], plot_bgcolor=C["surface"],
        font=dict(family=FONT, color=C["ink2"], size=12),
        xaxis=dict(ax, title=xtitle), yaxis=dict(ax, title=ytitle),
        margin=dict(l=60, r=20, t=48, b=48), height=height, hovermode="closest",
        legend=dict(orientation="h", yanchor="bottom", y=1.0, x=1.0, xanchor="right",
                    font=dict(size=11, color=C["ink2"]), bgcolor="rgba(0,0,0,0)"))
print("repo root:", ROOT)

## 1. Issue (before) — Non-Convex Functions and Big-M Binaries

`f(x)` is multimodal and non-convex. A convex solver cannot reach the minimum in a single pass, so we use piecewise linear approximation to fit it into an integer program.
Because the Big-M version places a selection binary `z_s` for each segment, **if there are K breakpoints, there are K−1 binaries**.
Below is a comparison of the number of binary variables between the SOS2 and Big-M versions.

In [ ]:
def count_bin(m):
    return sum(1 for v in m.getVars() if v.vtype() == "BINARY")

for method in ("sos2", "bigm"):
    m = pw.build_model(method)
    print(f"{method:5s}: 変数 {m.getNVars():3d} / バイナリ {count_bin(m):2d} / 折れ点 {pw.N_BREAK}")

## 2. Principle (principle) — Breakpoints, Approximation Error, and SOS2

A PWL approximation is a polygonal chain connecting the breakpoints `(x_k, f(x_k))`. As the number of segments increases, the approximation approaches the true curve.
Below, the true `f(x)` is overlaid with PWL approximations varying the number of breakpoints.

In [ ]:
def pwl_eval(xq, xs, ys):
    return np.interp(xq, xs, ys)

xq = np.linspace(pw.X_LO, pw.X_HI, 400)
ftrue = np.array([pw.f(v) for v in xq])

fig = go.Figure(layout=base_layout(
    "非凸関数 f(x) の区分線形近似(折れ点数を変える)", "x", "f(x)", height=420))
fig.add_trace(go.Scatter(x=xq, y=ftrue, mode="lines",
    line=dict(color=C["ink"], width=2.5), name="真の f(x)"))
for nb_pt, col in zip([5, 9, 21], [C["warn"], C["s2"], C["s1"]]):
    xs = np.linspace(pw.X_LO, pw.X_HI, nb_pt)
    ys = np.array([pw.f(v) for v in xs])
    fig.add_trace(go.Scatter(x=xs, y=ys, mode="lines+markers",
        line=dict(color=col, width=1.5, dash="dot"),
        marker=dict(color=col, size=6),
        name=f"PWL {nb_pt}点 (折れ点)"))
show(fig)

As the number of segments increases, the error decreases. We quantify the maximum approximation error as a function of the number of breakpoints.

In [ ]:
def max_err(nb_pt):
    xs = np.linspace(pw.X_LO, pw.X_HI, nb_pt)
    ys = np.array([pw.f(v) for v in xs])
    return float(np.max(np.abs(pwl_eval(xq, xs, ys) - ftrue)))

npts = list(range(3, 41, 2))
errs = [max_err(k) for k in npts]
bins_bigm = [k - 1 for k in npts]       # Big-M: バイナリ = 区分数 = K-1
bins_sos2 = [0 for _ in npts]           # SOS2: バイナリ 0

fig = make_subplots(specs=[[{"secondary_y": True}]])
fig.add_trace(go.Scatter(x=npts, y=errs, mode="lines+markers",
    line=dict(color=C["ink"], width=2.5), marker=dict(size=5),
    name="最大近似誤差"), secondary_y=False)
fig.add_trace(go.Scatter(x=npts, y=bins_bigm, mode="lines",
    line=dict(color=C["warn"], width=2, dash="dash"),
    name="Big-M のバイナリ数 (=K−1)"), secondary_y=True)
fig.add_trace(go.Scatter(x=npts, y=bins_sos2, mode="lines",
    line=dict(color=C["s1"], width=2),
    name="SOS2 のバイナリ数 (=0)"), secondary_y=True)
fig.update_layout(
    title=dict(text="精度 vs モデル規模: 折れ点を増やしても SOS2 はバイナリ0のまま",
               x=0.01, font=dict(color=C["ink"], size=15, family=FONT)),
    paper_bgcolor=C["surface"], plot_bgcolor=C["surface"],
    font=dict(family=FONT, color=C["ink2"], size=12), height=400,
    margin=dict(l=60, r=60, t=48, b=48), hovermode="x unified",
    legend=dict(orientation="h", yanchor="bottom", y=1.0, x=1.0, xanchor="right",
                font=dict(size=11, color=C["ink2"]), bgcolor="rgba(0,0,0,0)"))
fig.update_xaxes(title_text="折れ点数 K", gridcolor=C["grid"], linecolor=C["axis"], zeroline=False)
fig.update_yaxes(title_text="最大近似誤差", gridcolor=C["grid"], linecolor=C["axis"],
                 zeroline=False, secondary_y=False)
fig.update_yaxes(title_text="バイナリ変数の数", zeroline=False, secondary_y=True)
show(fig)

**SOS2 represents adjacency without binaries.** The weights $\lambda_k$ ($\sum_k\lambda_k=1,\ \lambda\ge0$) in PWL must inherently be "non-zero for at most two adjacent points". Big-M forces this adjacency using segment binaries $z_s$ and $\lambda_k \le z_{k-1}+z_k$. An **SOS2 constraint** (Special Ordered Set type 2: at most two non-zeros, and they must be adjacent) passes this exact property directly to SCIP. Because SCIP has specialized branching rules for SOS2, the model is lighter since it bypasses auxiliary binaries. As shown in the figure above, even when the number of breakpoints increases, the number of binaries in SOS2 remains 0.

## 3. Application (how) — `mk.pwl_sos2`

Any single-variable function can be replaced with SOS2-PWL just by passing the breakpoints and their function values.

```python
xs = [x0, x1, ..., xK]          # Breakpoints (ascending)
ys = [f(x0), f(x1), ..., f(xK)] # Function values at each breakpoint
y = mk.pwl_sos2(m, x, xs, ys, name="fx")  # y ≈ f(x). Uses neither binaries nor Big-M
```

Below is a minimal working check. We approximate $f(x)=x^2$ with 3 points and get the value at $x=1$.

In [ ]:
from pyscipopt import Model
m = Model(); m.hideOutput()
x = m.addVar(lb=0.0, ub=2.0, name="x")
y = mk.pwl_sos2(m, x, [0.0, 1.0, 2.0], [0.0, 1.0, 4.0], "sq")
m.addCons(x == 1.0)
m.setObjective(y, "minimize")
m.optimize()
print(f"x=1 での近似値 y = {m.getVal(y):.2f}  (f(1)=1.0)")

## 4. Effect (before/after) — SOS2 vs Big-M

We solve the SOS2 and Big-M versions with the same breakpoints and compare the optimal value, model scale, and solving time.

In [ ]:
df = mk.compare_variants(
    {"bigm": lambda: pw.build_model("bigm"),
     "sos2": lambda: pw.build_model("sos2")}, time_limit=10)

opt = {mth: pw.build_model(mth) for mth in ("bigm", "sos2")}
for mth, m in opt.items():
    m.hideOutput(); m.optimize()
bins = {mth: count_bin(pw.build_model(mth)) for mth in ("bigm", "sos2")}
print("最適値 bigm =", f"{opt['bigm'].getObjVal():.4f}",
      "/ sos2 =", f"{opt['sos2'].getObjVal():.4f}  (一致)")
df[["variant", "root_dual", "final_dual", "final_gap", "nodes", "time"]]

In [ ]:
labels = ["Big-M", "SOS2"]
bar_colors = [C["warn"], C["s1"]]
fig = make_subplots(rows=1, cols=2, horizontal_spacing=0.18,
    subplot_titles=("バイナリ変数の数 (少ないほど軽い)", "探索ノード数"))
fig.add_trace(go.Bar(x=labels, y=[bins["bigm"], bins["sos2"]], marker_color=bar_colors,
    text=[bins["bigm"], bins["sos2"]], textposition="outside", cliponaxis=False,
    showlegend=False), row=1, col=1)
nodes = [int(df.loc[df.variant == v, "nodes"].iloc[0]) for v in ("bigm", "sos2")]
fig.add_trace(go.Bar(x=labels, y=nodes, marker_color=bar_colors,
    text=nodes, textposition="outside", cliponaxis=False, showlegend=False), row=1, col=2)
fig.update_layout(paper_bgcolor=C["surface"], plot_bgcolor=C["surface"],
    font=dict(family=FONT, color=C["ink2"], size=12), height=360,
    margin=dict(l=40, r=20, t=64, b=40),
    title=dict(text="SOS2 vs Big-M の before / after", x=0.01,
               font=dict(color=C["ink"], size=15)))
fig.update_yaxes(gridcolor=C["grid"], linecolor=C["axis"], zeroline=False)
fig.update_xaxes(linecolor=C["axis"])
show(fig)

## Summary

- The SOS2 version and Big-M version reach the **same optimal value** (they are equivalent PWL representations).
- The difference lies in **model scale**. For Big-M, binaries increase in proportion to the number of breakpoints, while SOS2 represents the same adjacency with 0 binary variables. Since SCIP possesses branching rules exclusively for SOS2, the search is lighter as it skips auxiliary binaries. In this small example, both can be solved immediately, but when PWL is embedded into many variables in a model, this difference in variable counts has a significant impact.

### When It Is Ineffective / Cautions

- **Exclusive to single-variable functions**. It cannot be used for multi-variable non-convex terms (like products) → in such cases, consider [Exact Linearization of Integer × Continuous](01_linearize_product.ipynb) or convex hull reformulation.
- While increasing breakpoints improves approximation precision, the number of $\lambda$ variables also increases (a trade-off with precision). Use the "Precision vs Model Scale" chart above to choose a number of breakpoints appropriate for the required precision.

Related: [Methods Guide 2. PWL Approximation (SOS2)](../../playbook/02-pwl-sos2.md) /
API [`mk.pwl_sos2`](../../api/transforms.md)